# 04 — Corridor Comparison

Mục tiêu notebook này:

- So sánh **15 corridor** thuộc 3 khu đã validated: **Bình Thạnh, Bình Tân, Quận 12**.
- Không dùng weighted score và không tự đặt trọng số.
- Không biến `traffic evidence count` thành traffic volume.
- Không biến số anchor thành customer count.
- Chỉ dùng rent median khi có **>= 3 clean current commercial-usable samples 25–45 m²**.
- Các listing mới nhưng chưa verify exact unit/address chỉ dùng để nâng mức **research readiness**, chưa khóa median.

Kết quả notebook là:
1. Bảng comparison master.
2. Trạng thái readiness theo từng corridor.
3. Shortlist để đi **site verification**.
4. Bảng tổng hợp theo district.
5. Export ra `Corridor_Comparison_Output.xlsx`.

> Lưu ý: bước này chọn **corridor để đi site-level**, chưa chọn số nhà cuối cùng.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

## 1. Tìm `Data_input.xlsx`

Notebook hỗ trợ vài cấu trúc folder thường dùng trong project. Nếu file ở vị trí khác, sửa `CANDIDATES`.

In [ ]:
CANDIDATES = [
    Path("../Data/Raw/Data_input.xlsx"),
    Path("../Data/Data_input.xlsx"),
    Path("Data/Raw/Data_input.xlsx"),
    Path("Data_input.xlsx"),
    Path("/mnt/data/Data_input.xlsx"),
]

DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy Data_input.xlsx. Hãy sửa danh sách CANDIDATES ở cell này."
    )

print("Using:", DATA_PATH.resolve())

## 2. Đọc các sheet cần thiết

Notebook ưu tiên sheet tổng hợp mới nhất `26_Corridor_Consolidated`.  
Nếu chưa có, notebook fallback về `23_Corridor_Research_Status` + `22_Corridor_Rent_Summary`.

In [ ]:
xls = pd.ExcelFile(DATA_PATH)
sheet_names = xls.sheet_names

def find_sheet(prefix):
    matches = [s for s in sheet_names if s == prefix or s.startswith(prefix)]
    if not matches:
        return None
    return matches[0]

required_base = {
    "corridor": find_sheet("15_Top3_Corridor_Research"),
    "status": find_sheet("23_Corridor_Research_Status"),
    "rent": find_sheet("22_Corridor_Rent_Summary"),
}
optional = {
    "consolidated": find_sheet("26_Corridor_Consolidated"),
    "sufficiency": find_sheet("36_Evidence_Sufficiency"),
    "research_master": find_sheet("24_Corridor_Research_Master"),
    "method_quality": find_sheet("25_Method_Quality"),
}

print("Resolved sheets:")
for k,v in {**required_base, **optional}.items():
    print(f"  {k:16s}: {v}")

missing = [k for k,v in required_base.items() if v is None]
if missing:
    raise ValueError(f"Thiếu sheet bắt buộc: {missing}")


In [ ]:
corridor = pd.read_excel(DATA_PATH, sheet_name=required_base["corridor"])
status = pd.read_excel(DATA_PATH, sheet_name=required_base["status"])
rent = pd.read_excel(DATA_PATH, sheet_name=required_base["rent"])

consolidated = (
    pd.read_excel(DATA_PATH, sheet_name=optional["consolidated"])
    if optional["consolidated"] else None
)
sufficiency = (
    pd.read_excel(DATA_PATH, sheet_name=optional["sufficiency"])
    if optional["sufficiency"] else None
)
research_master = (
    pd.read_excel(DATA_PATH, sheet_name=optional["research_master"])
    if optional["research_master"] else None
)

print("corridor:", corridor.shape)
print("status:", status.shape)
print("rent:", rent.shape)
print("consolidated:", None if consolidated is None else consolidated.shape)
print("sufficiency:", None if sufficiency is None else sufficiency.shape)
print("research_master:", None if research_master is None else research_master.shape)


## 3. Kiểm tra scope

Scope đúng phải là **15 corridor = 5 tuyến × 3 district**.

In [ ]:
expected_districts = {"Bình Thạnh", "Bình Tân", "Quận 12"}

scope = corridor[["district", "corridor"]].drop_duplicates().copy()

print("Districts:", sorted(scope["district"].dropna().unique()))
print("Corridors:", len(scope))
print(scope.groupby("district").size())

assert set(scope["district"].dropna()) == expected_districts, "District scope không đúng Top 3."
assert len(scope) == 15, f"Expected 15 corridors, got {len(scope)}"
assert (scope.groupby("district").size() == 5).all(), "Mỗi district phải có đúng 5 corridors."

## 4. Build comparison master

`traffic_evidence_rows` và `traffic_source_count` chỉ phản ánh **evidence breadth**.

Không dùng chúng như:
- lượt xe/ngày;
- AADT;
- traffic score.

Tương tự, `context_anchor_rows` là độ rộng evidence, không phải số khách.

In [ ]:
if consolidated is not None:
    master = consolidated.copy()
else:
    master = (
        status.merge(
            rent[[
                "district","corridor","clean_current_usable_n",
                "median_rent_million_month","rent_readiness",
                "target_rent_15_20m_context"
            ]],
            on=["district","corridor"],
            how="left",
            suffixes=("","_rent")
        )
    )
    master["locked_clean_rent_n"] = master["clean_current_usable_n"]
    master["locked_rent_median_million"] = master["median_rent_million_month"]
    master["decision_use"] = np.where(
        master["research_completeness"].eq("HIGH"),
        "READY_FOR_CORRIDOR_COMPARISON",
        "KEEP_IN_COMPARISON_WITH_DATA_GUARDRAIL"
    )
    master["post_research_status"] = master["research_completeness"]
    master["new_research_addition"] = ""
    master["method_guardrail"] = (
        "No weighted score. Evidence breadth is not traffic volume or customer count."
    )

# add locked rent context if not already present
rent_small = rent[[
    "district","corridor","clean_current_usable_n",
    "median_rent_million_month","rent_readiness",
    "target_rent_15_20m_context"
]].copy()

master = master.merge(
    rent_small,
    on=["district","corridor"],
    how="left",
    suffixes=("","_sheet22")
)

master["locked_clean_rent_n"] = master["locked_clean_rent_n"].fillna(
    master["clean_current_usable_n"]
)
master["locked_rent_median_million"] = master["locked_rent_median_million"].fillna(
    master["median_rent_million_month"]
)

master = scope.merge(master, on=["district","corridor"], how="left")
print(master.shape)
master.head()

# Add audited evidence sufficiency / observed competition if available.
if sufficiency is not None:
    suff_cols = [
        c for c in [
            "district","corridor","traffic_layer","anchor_layer","rent_layer",
            "risk_or_infra_layer","cross_source_quality",
            "current_research_readiness","remaining_gap",
            "recommended_next_step","note",
            "observed_direct_repair_points_n","competition_observation","audit_status"
        ]
        if c in sufficiency.columns
    ]
    suff_small = sufficiency[suff_cols].copy()

    # Avoid collisions with fields already built later in this notebook.
    rename_map = {}
    for c in ["traffic_layer","anchor_layer","rent_layer","remaining_gap"]:
        if c in suff_small.columns:
            rename_map[c] = c + "_audit"
    suff_small = suff_small.rename(columns=rename_map)

    master = master.merge(
        suff_small,
        on=["district","corridor"],
        how="left"
    )


## 5. Chuẩn hóa trạng thái để so sánh

Không cộng điểm.

Mỗi corridor được nhìn qua các lớp độc lập:

- **Traffic evidence layer**
- **Catchment / anchor layer**
- **Rent layer**
- **Format/supply constraint**
- **Research readiness**

Các trạng thái này dùng để quyết định corridor nào đủ chắc để đi site-level trước.

In [ ]:
def traffic_layer(row):
    n = row.get("traffic_evidence_rows", np.nan)
    if pd.isna(n):
        return "TRAFFIC_EVIDENCE_UNKNOWN"
    return "TRAFFIC_EVIDENCE_ESTABLISHED" if n >= 2 else "TRAFFIC_EVIDENCE_LIMITED"

def anchor_layer(row):
    n = row.get("existing_context_anchor_rows", row.get("context_anchor_rows", np.nan))
    if pd.isna(n):
        return "ANCHOR_EVIDENCE_UNKNOWN"
    if n >= 2:
        return "ANCHOR_EVIDENCE_BROAD"
    if n == 1:
        return "ANCHOR_EVIDENCE_PRESENT"
    return "ANCHOR_EVIDENCE_LIMITED"

def rent_layer(row):
    n = row.get("locked_clean_rent_n", np.nan)
    post = str(row.get("post_research_status", "")).upper()
    suff_rent = str(row.get("rent_layer_audit", "")).upper()

    if pd.notna(n) and n >= 3:
        return "RENT_LOCKED_N_GE_3"

    # Structural format mismatch must not be mislabeled as generic missing data.
    mismatch_tokens = [
        "FORMAT_MISMATCH",
        "TARGET_FORMAT_SUPPLY_GAP",
        "STRUCTURAL_FORMAT_MISMATCH",
    ]
    if any(t in post for t in mismatch_tokens) or any(t in suff_rent for t in mismatch_tokens):
        return "TARGET_FORMAT_SUPPLY_MISMATCH"

    if "NEAR_READY" in post:
        return "RENT_NEAR_READY_VERIFY"

    if pd.notna(n) and n > 0:
        return "RENT_PARTIAL"

    if "OUTLIER" in post or "OUTLIER" in suff_rent:
        return "RENT_OUTLIER_UNRESOLVED"

    return "RENT_DATA_LIMITED"

def budget_context(row):
    med = row.get("locked_rent_median_million", np.nan)
    if pd.isna(med):
        return "RENT_MEDIAN_NOT_LOCKED"
    if med <= 20:
        return "WITHIN_OR_BELOW_20M_BUDGET_CONTEXT"
    return "ABOVE_20M_BUDGET_CONTEXT"

master["traffic_layer"] = master.apply(traffic_layer, axis=1)
master["anchor_layer"] = master.apply(anchor_layer, axis=1)
master["rent_layer"] = master.apply(rent_layer, axis=1)
master["budget_context"] = master.apply(budget_context, axis=1)


## 6. Shortlist rule cho site verification

Notebook **không ép** mỗi district phải ra đúng 2 tuyến nếu evidence chưa đủ.

Rule:

### Tier A — `SITE_VERIFY_PRIORITY`
Corridor đã `READY_FOR_CORRIDOR_COMPARISON`.

### Tier B — `SITE_VERIFY_AFTER_RENT_CHECK`
Corridor đang `NEAR_READY_PENDING_LISTING_VERIFICATION`.

### Tier C — `KEEP_WITH_GUARDRAIL`
Traffic/catchment có ý nghĩa nhưng rent hoặc format supply còn thiếu.

Điểm quan trọng: đây là **evidence-readiness tier**, không phải attractiveness score.

In [ ]:
def shortlist_tier(row):
    decision = str(row.get("decision_use",""))
    post = str(row.get("post_research_status","")).upper()
    audit_readiness = str(row.get("current_research_readiness","")).upper()
    rent_state = str(row.get("rent_layer","")).upper()

    if "TARGET_FORMAT_SUPPLY_MISMATCH" in rent_state or "STRUCTURAL_FORMAT_MISMATCH" in post:
        return "RESERVE_FORMAT_MISMATCH"

    if decision == "READY_FOR_CORRIDOR_COMPARISON":
        return "SITE_VERIFY_PRIORITY"

    if audit_readiness in {"MEDIUM_HIGH", "MEDIUM_PLUS"}:
        return "KEEP_WITH_TARGETED_GAP"

    return "KEEP_WITH_GUARDRAIL"

master["shortlist_tier"] = master.apply(shortlist_tier, axis=1)

tier_order = {
    "SITE_VERIFY_PRIORITY": 1,
    "KEEP_WITH_TARGETED_GAP": 2,
    "KEEP_WITH_GUARDRAIL": 3,
    "RESERVE_FORMAT_MISMATCH": 4,
}
master["tier_order"] = master["shortlist_tier"].map(tier_order)

print(
    master.groupby(["district","shortlist_tier"])
          .size()
          .rename("corridors")
)


## 7. Comparison table

Không sort theo "điểm". Chỉ group theo district và tier readiness.

In [ ]:
display_cols = [
    "district","corridor","shortlist_tier",
    "traffic_evidence_rows","traffic_sources",
    "existing_context_anchor_rows","existing_context_types",
    "locked_clean_rent_n","locked_rent_median_million",
    "traffic_layer","anchor_layer","rent_layer","budget_context",
    "risk_or_infra_layer","cross_source_quality",
    "observed_direct_repair_points_n","competition_observation",
    "current_research_readiness","audit_status",
    "post_research_status","remaining_gap","recommended_next_step",
    "new_research_addition"
]

available_cols = [c for c in display_cols if c in master.columns]

sort_cols = ["district"]
if "tier_order" in master.columns:
    sort_cols.append("tier_order")

comparison = (
    master.sort_values(sort_cols)
          .loc[:, available_cols]
          .reset_index(drop=True)
)

comparison


## 8. Site-verification shortlist

Shortlist này lấy:

- Tier A: đi site-level trước.
- Tier B: verify listing/rent trước rồi mới đi site-level.

Không lấy Tier C vào shortlist chính, nhưng vẫn giữ làm reserve.

In [ ]:
site_shortlist = master[
    master["shortlist_tier"].isin([
        "SITE_VERIFY_PRIORITY",
        "KEEP_WITH_TARGETED_GAP"
    ])
].copy()

site_shortlist = site_shortlist.sort_values(
    ["district","tier_order","corridor"]
)

print(site_shortlist.groupby(["district","shortlist_tier"]).size())

cols = [
    "district","corridor","shortlist_tier",
    "locked_clean_rent_n","locked_rent_median_million",
    "observed_direct_repair_points_n","competition_observation",
    "risk_or_infra_layer","current_research_readiness",
    "post_research_status","remaining_gap","recommended_next_step"
]
site_shortlist[[c for c in cols if c in site_shortlist.columns]]


### Cách đọc shortlist

Nếu một district còn >2 tuyến ở Tier A, **không nên ép xuống 2 bằng score tự đặt**.

Lúc đó bước đúng là đi site-level và dùng dữ liệu địa chỉ thật:

- frontage;
- parking;
- visibility;
- đoạn đường cụ thể;
- giao lộ;
- hướng lưu thông;
- vị trí đối thủ;
- khoảng cách iCare;
- giá thuê thực tế;
- diện tích thực tế;
- khả năng dừng xe.

Sau đó mới loại tiếp.

## 9. District summary

In [ ]:
district_summary = (
    master.groupby("district", dropna=False)
    .agg(
        corridors=("corridor","nunique"),
        site_verify_priority=("shortlist_tier", lambda s: (s=="SITE_VERIFY_PRIORITY").sum()),
        targeted_gap=("shortlist_tier", lambda s: (s=="KEEP_WITH_TARGETED_GAP").sum()),
        guarded=("shortlist_tier", lambda s: (s=="KEEP_WITH_GUARDRAIL").sum()),
        reserve_format_mismatch=("shortlist_tier", lambda s: (s=="RESERVE_FORMAT_MISMATCH").sum()),
        locked_rent_corridors=("locked_clean_rent_n", lambda s: (pd.to_numeric(s, errors="coerce") >= 3).sum()),
    )
    .reset_index()
)

district_summary


## 10. Guardrails để trình bày với sếp

- Không gọi `traffic_evidence_rows` là traffic volume.
- Không gọi `context_anchor_rows` là demand/customer count.
- Rent median chỉ dùng khi `locked_clean_rent_n >= 3`.
- Median thấp hơn 15m không phải là điểm trừ; target 15–20m là budget context.
- Corridor thiếu rent có thể là **data gap** hoặc **target-format supply gap**.
- Không dùng weighted score.
- Không buộc đủ 2 tuyến/district bằng cách nới rule.

In [ ]:
guardrails = pd.DataFrame({
    "rule": [
        "No weighted score",
        "Readiness tier != corridor attractiveness rank",
        "Traffic evidence count != traffic volume",
        "Anchor count != customer demand",
        "Observed competitor points != total competitor count or demand",
        "Missing local-business search result != zero competitors",
        "Locked rent median requires >=3 clean current commercial-usable 25–45m² samples",
        "Missing rent != 0",
        "Below-target rent is not automatically bad",
        "Unverified / near-road / office listing does not enter locked storefront median",
        "Format mismatch is kept as a structural risk, not filled with fake rent",
        "Final site selection requires address-level field verification",
    ]
})
guardrails


## 11. Export

File output dùng cho bước tiếp theo:

`Corridor_Comparison_Output.xlsx`

In [ ]:
OUTPUT_CANDIDATES = [
    Path("../Data/Processed/Corridor_Comparison_Output.xlsx"),
    Path("Data/Processed/Corridor_Comparison_Output.xlsx"),
    Path("Corridor_Comparison_Output.xlsx"),
]

OUTPUT_PATH = OUTPUT_CANDIDATES[0]
try:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
except Exception:
    OUTPUT_PATH = OUTPUT_CANDIDATES[-1]

comparison_export = master.drop(columns=["tier_order"], errors="ignore").copy()

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    comparison_export.to_excel(writer, sheet_name="01_Corridor_Master", index=False)
    comparison.to_excel(writer, sheet_name="02_Comparison_Readiness", index=False)
    site_shortlist.to_excel(writer, sheet_name="03_Site_Verify_Shortlist", index=False)
    district_summary.to_excel(writer, sheet_name="04_District_Summary", index=False)
    guardrails.to_excel(writer, sheet_name="05_Guardrails", index=False)

print("Exported:", OUTPUT_PATH.resolve())

## 12. Kết luận của notebook

Notebook này **không tự chọn địa chỉ cuối cùng**.

Nó đóng research broad và chuyển project sang:

**corridor comparison → site verification → site-level shortlist → final site.**